In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -qU langchain langchain-community langchain-core langchain-huggingface chromadb sentence-transformers
!pip install -qU transformers accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 111.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.1/494.1 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 108.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.4/157.4 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 119.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6

In [2]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. 경로 설정
LOCAL_MODEL_PATH = "/content/drive/MyDrive/DILAB/Models/Qwen2-7B-Instruct"

# 벡터 DB가 저장된 경로
BASE_DIR = "/content/drive/MyDrive/DILAB/YB/DILAB/LCK_RAG_Experiment"
DB_PATH = os.path.join(BASE_DIR, "VectorDB")

# 2. 벡터 DB 로드
print(f"[백터 DB 경로]: {DB_PATH}")

embedding_model = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

vector_db = Chroma(persist_directory=DB_PATH, embedding_function=embedding_model)
print("벡터 DB 로드 완료")

# 3. 로컬 Qwen 모델 로드
print(f"[로컬 모델 경로]: {LOCAL_MODEL_PATH}")

if not os.path.exists(LOCAL_MODEL_PATH):
    print(f"{LOCAL_MODEL_PATH}")
else:
    try:
        # 1. 토크나이저 로드
        tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)

        # 2. 양자화 설정 (4bit 로딩을 위한 설정 객체 생성)
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16
        )

        # 3. 모델 로드
        model = AutoModelForCausalLM.from_pretrained(
            LOCAL_MODEL_PATH,
            device_map="auto",
            quantization_config=bnb_config,
            dtype=torch.float16
        )

        # 4. 파이프라인 구축
        text_generator = pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer,
            max_new_tokens=512,
            temperature=0.1
        )
        print("Qwen2-7B-Instruct 모델 로드 완료")

    except Exception as e:
        print(e)

[백터 DB 경로]: /content/drive/MyDrive/DILAB/YB/DILAB/LCK_RAG_Experiment/VectorDB


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: jhgan/ko-sroberta-multitask
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipython-input-2137132214.py:23: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_db = Chroma(persist_directory=DB_PATH, embedding_function=embedding_model)


벡터 DB 로드 완료
[로컬 모델 경로]: /content/drive/MyDrive/DILAB/Models/Qwen2-7B-Instruct


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Qwen2-7B-Instruct 모델 로드 완료


In [27]:
import warnings
from transformers import logging

# 0. 경고 메시지 차단
warnings.filterwarnings("ignore")
logging.set_verbosity_error()

def experiment_rag_comparison(question):
    print(f"\n{'='*80}")
    print(f"[질문]: {question}")
    print(f"{'='*80}\n")

    # [Case 1] 순수 Qwen (RAG 미적용)
    print("[1. RAG 적용 전 (Original Qwen)]")

    messages_no_rag = [
        {"role": "system", "content": "당신은 e스포츠 전문가입니다. 질문에 답변해 주세요."},
        {"role": "user", "content": question}
    ]
    prompt_no_rag = tokenizer.apply_chat_template(messages_no_rag, tokenize=False, add_generation_prompt=True)

    output_no_rag = text_generator(
        prompt_no_rag,
        max_new_tokens=512,
        max_length=None,
        return_full_text=False
    )
    print(f"\n답변:\n{output_no_rag[0]['generated_text'].strip()}")
    print(f"\n{'-'*80}\n")

    # [Case 2] RAG 적용
    print("[2. RAG 적용 후 (With Context)]")

    raw_docs = vector_db.similarity_search(question, k=10) # 넉넉히 10개 가져옴
    docs = []
    seen_titles = set()

    for doc in raw_docs:
        title = doc.metadata.get("title", "제목 없음")
        if title in seen_titles:
            continue
        seen_titles.add(title)
        docs.append(doc)
        if len(docs) >= 5:
            break

    context_text = ""
    print(f"\n[검색된 근거 자료]")
    if not docs:
        print("검색된 문서 없음")
    else:
        for i, doc in enumerate(docs):
            title = doc.metadata.get("title", "제목 없음")
            date = doc.metadata.get("date", "날짜 미상")
            preview = doc.page_content.replace("\n", " ").strip()[:80]
            print(f"      [{i+1}] {title} ({date}) -> {preview}...")
            context_text += f"문서{i+1}: {doc.page_content}\n\n"

    # 2. 프롬프트 구성
    system_prompt = f"""
    당신은 e스포츠 분석가입니다.
    아래 [참고 자료]에 있는 내용만을 근거로 질문에 답변하세요.

    [지침]
    1. '참고 자료'에 없는 내용은 절대 지어내지 마세요.
    2. 질문과 관련 없는 내용은 배제하고 핵심만 답변하세요.

    [참고 자료]
    {context_text}
    """

    messages_rag = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]
    prompt_rag = tokenizer.apply_chat_template(messages_rag, tokenize=False, add_generation_prompt=True)

    # 3. 실제 입력된 프롬프트 출력
    print(f"\n[LLM에게 실제로 들어가는 입력 데이터 (Prompt)]")
    print(f"   {prompt_rag[:300]} \n   ... (중략: 기사 본문들) ... \n   {prompt_rag[-200:]}")

    # 4. 답변 생성
    output_rag = text_generator(
        prompt_rag,
        max_new_tokens=1024,
        max_length=None,
        return_full_text=False
    )
    print(f"\n답변:\n{output_rag[0]['generated_text'].strip()}")
    print(f"\n{'='*80}")

# 실행
experiment_rag_comparison("2026년 LCK CUP에서 DRX팀의 경기 결과를 알려줘")


[질문]: 2026년 LCK CUP에서 DRX팀의 경기 결과를 알려줘

[1. RAG 적용 전 (Original Qwen)]

답변:
죄송합니다, 하지만 현재 시점에서 2026년 LCK CUP의 DRX 팀의 경기 결과를 예측하는 것은 불가능합니다. 스포츠 경기 결과는 여러 가지 요인에 의해 결정되며, 이 중에는 선수들의 상태, 상대 팀의 능력, 경기 날짜와 장소, 그리고 기상 조건 등이 포함됩니다. 또한, 이 모든 요인은 미래에 어떻게 변할지 알 수 없습니다. 따라서, 특정 팀의 미래 경기 결과를 정확하게 예측하는 것은 불가능합니다.

--------------------------------------------------------------------------------

[2. RAG 적용 후 (With Context)]

[검색된 근거 자료]
      [1] PO 직행·탈락 가르는 승부처…2026 LCK컵 3주 차 관전 포인트는 (2026-01-24) -> 제목: PO 직행·탈락 가르는 승부처…2026 LCK컵 3주 차 관전 포인트는...
      [2] 2026 LCK컵 2주 차, T1-kt 롤스터 맞대결… ‘롤드컵 결승 리매치’ 성사 (2026-01-20) -> 제목: 2026 LCK컵 2주 차, T1-kt 롤스터 맞대결… ‘롤드컵 결승 리매치’ 성사...
      [3] T1, 2026 LCK컵 2연승, 풀세트 혈투 끝 DRX 제압... 바론 그룹 반등 신호탄 (2026-01-18) -> 제목: T1, 2026 LCK컵 2연승, 풀세트 혈투 끝 DRX 제압... 바론 그룹 반등 신호탄...
      [4] 조재읍 “당장의 승패에 연연하지 않겠다” (2026-01-21) -> 내용: DRX 조재읍 감독이 당장의 승패에 연연하지 않고 멀리 내다보겠다고 말했다. DRX는 21일 서울 종로구 LCK 아레나에서 열린 2026...
      [5] DRX, 농심과 난타전 끝 2:1 승…‘안딜’ 궁 한방으로 마침표 [LCK컵] (종합) (2026